# 02 – Nettoyage des données

Objectif : produire une version propre du jeu de données, utilisable pour les agrégations.
Chaque décision s'appuie sur un constat de `01_exploration.ipynb` et est documentée
(quoi / pourquoi / nombre de lignes touchées).

Entrée : `data/raw/Accidental_Drug_Related_Deaths_2012-2022.csv` (jamais modifié)
Sortie : `data/processed/` (fichier nettoyé)

## Résumé des décisions de nettoyage

| # | Problème constaté | Solution appliquée | Impact |
|---|---|---|---|
| 1 | `Other`, `Other Opioid` et `Ethnicity` inexploitables (texte libre, 85 % vide) | Suppression des colonnes | 3 colonnes, 0 ligne |
| 2 | Substances notées en texte (`Y`, `y`, `Y POPS`, `N`, `P`, vides…) | Conversion en booléens (`src/nettoyage.py`) | 21 colonnes, 48 détections préservées |
| 3 | Dates stockées en texte (MM/JJ/AAAA) | `to_datetime` avec format explicite | 10 654 dates, 0 échec |
| 4 | Sexe manquant | Remplacé par `Unknown` | 9 lignes |
| 5 | Âge manquant | Remplacé par la médiane (43 ans) | 2 lignes |
| 5 bis | Âge au-dessus de la borne haute (87 ans) | Conservé : valeur plausible | 1 ligne |
| 5 ter | Âge stocké en décimal après imputation | Conversion en entier (`astype(int)`) | 0 valeur modifiée |
| 6 | Nature du décès : casse incohérente | `str.title()` | 14 lignes |
| 6 bis | Nature du décès : vides | Remplacés par `Unknown` | 9 lignes |
| 6 ter | Décès `Natural`, hors du périmètre du jeu | Ligne supprimée | **1 ligne supprimée** |
| 7 | Lieu : formulaire modifié en 2021, apostrophes, faute de frappe | Nouvelle colonne `Lieu` en 5 groupes | 10 653 lignes, dont 22 `Unknown` |
| 8 | Doublons possibles | Vérification avec `duplicated` | 0 doublon |

**Bilan : 10 654 → 10 653 lignes, 48 → 46 colonnes.**

In [1]:
import pandas as pd

import sys
sys.path.append("..")
from src.nettoyage import convertir_en_booleens

deces = pd.read_csv("../data/raw/Accidental_Drug_Related_Deaths_2012-2022.csv")
nb_lignes_depart = len(deces)
print(f"Lignes au départ : {nb_lignes_depart}, colonnes : {deces.shape[1]}")

Lignes au départ : 10654, colonnes : 48


**Décision 1 – Suppression des colonnes écartées**
- Quoi : suppression de `Other`, `Other Opioid` et `Ethnicity` (3 colonnes, aucune ligne).
- Pourquoi : `Other` = 147 valeurs en texte libre, même substance sous plusieurs écritures (constat 12) ;
  `Other Opioid` = 39 noms de médicaments, dont des non-opioïdes (constat 4) ;
  `Ethnicity` = 85 % de vides, non utilisée par nos questions (constat 2).
- Les autres colonnes en texte libre sont gardées car elles ne gênent pas l'analyse.

In [2]:
colonnes_ecartees = ["Other", "Other Opioid", "Ethnicity"]
deces = deces.drop(columns=colonnes_ecartees)
print(f"Colonnes après suppression : {deces.shape[1]}")

Colonnes après suppression : 45


**Décision 2 – Conversion des substances en booléens**
- Quoi : les colonnes de `Heroin` à `Any Opioid` passent en True (détectée) / False (non détectée).
- Pourquoi : pour compter les détections avec `sum`. Valeurs considérées comme détectées :
  `Y`, `y`, `Y POPS`, `Y (PTCH)`, soit 48 détections qu'un simple `== "Y"` aurait perdues.
  Tout le reste est considéré comme non détecté : vides, `N` (8 dans `Any Opioid`),
  `P` (1 dans `Ethanol`, ambigu) et les 3 notes en texte libre de `Morphine (Not Heroin)` (constat 4).

In [3]:
colonnes_substances = deces.loc[:, "Heroin":"Any Opioid"].columns
print(f"Nombre de colonnes de substances : {len(colonnes_substances)}")
deces = convertir_en_booleens(deces, colonnes_substances)
print(deces[colonnes_substances].dtypes.value_counts())

Nombre de colonnes de substances : 21
bool    21
Name: count, dtype: int64


**Décision 3 – Conversion des dates**
- Quoi : la colonne `Date` passe du texte au type date (format MM/JJ/AAAA).
- Pourquoi : pour extraire l'année et le mois (questions 1 et 3, étape 07).
- Toutes les dates sont gardées, y compris 2015 et 2017 où 100 % sont des dates de déclaration
  (`Date Type`) : l'analyse par année reste valable, l'analyse par mois est moins précise pour ces années (constat 5).

In [4]:
deces["Date"] = pd.to_datetime(deces["Date"], format="%m/%d/%Y", errors="coerce")
print(f"Dates non converties : {deces['Date'].isna().sum()}")
print(f"Période couverte : du {deces['Date'].min()} au {deces['Date'].max()}")

Dates non converties : 0
Période couverte : du 2012-01-01 00:00:00 au 2022-12-31 00:00:00


**Décision 4 – Sexe manquant**
- Quoi : les 9 valeurs vides de `Sex` sont remplacées par `Unknown`.
- Pourquoi : les lignes restent utiles pour les autres questions (substances, lieu, comté),
  on ne les supprime donc pas. La catégorie `Unknown` ne sera pas affichée dans les graphiques
  sur le sexe (question 3) (constat 6).

In [5]:
nb_sexe_vides = deces["Sex"].isna().sum()
deces["Sex"] = deces["Sex"].fillna("Unknown")
print(f"Valeurs de sexe remplacées : {nb_sexe_vides}")
print(deces["Sex"].value_counts())

Valeurs de sexe remplacées : 9
Sex
Male       7890
Female     2755
Unknown       9
Name: count, dtype: int64


**Décision 5 – Âge : manquants et valeurs extrêmes**
- Manquants : les 2 âges vides sont remplacés par la médiane (43 ans).
  Médiane plutôt que moyenne car elle est peu sensible aux valeurs extrêmes.
  2 lignes sur 10 654 : effet négligeable sur les résultats (constat 8).
- Valeurs extrêmes : règle des 1,5 écarts interquartiles, bornes 1,5 et 85,5 ans.
  Un seul âge hors bornes (87 ans), conservé : il est plausible, ce n'est pas une erreur de saisie.

In [6]:
nb_ages_vides = deces["Age"].isna().sum()
mediane_age = deces["Age"].median()
deces["Age"] = deces["Age"].fillna(mediane_age)
print(f"Âges remplacés : {nb_ages_vides} par la médiane {mediane_age}")
print(f"Âges encore manquants : {deces['Age'].isna().sum()}")

Âges remplacés : 2 par la médiane 43.0
Âges encore manquants : 0


**Décision 5 ter : type de la colonne `Age`**
- **Quoi** : conversion de `Age` de `float64` (décimal) en entier.
- **Pourquoi** : la colonne contenait 2 valeurs manquantes (NaN), que pandas ne peut stocker que dans une colonne décimale. Après l'imputation, il ne reste aucun manquant, et un âge s'exprime en années entières.
- **Lignes touchées** : aucune valeur modifiée, seul le type change (contrôle ci-dessous : aucun âge avec une partie décimale).

In [7]:
print(f"Âges non entiers : {(deces['Age'] % 1 != 0).sum()}")
deces["Age"] = deces["Age"].astype(int)

Âges non entiers : 0


**Décision 6 – Nature du décès**
- Harmonisation de la casse avec `str.title()` : `accident` (13) et `ACCIDENT` (1) deviennent `Accident`.
- Vides (9) → `Unknown`.
- Suppression de la ligne `Natural` (1) : un décès naturel sort de la définition du jeu
  (décès **accidentels**). C'est la seule ligne supprimée du nettoyage.
- `Pending` (14) conservés : enquête non close au moment de l'extraction (limite) (constat 10).

In [8]:
deces["Manner of Death"] = deces["Manner of Death"].str.title()
deces["Manner of Death"] = deces["Manner of Death"].fillna("Unknown")
deces = deces[deces["Manner of Death"] != "Natural"]
print(deces["Manner of Death"].value_counts())
print(f"Lignes : {nb_lignes_depart} au départ, {len(deces)} maintenant")

Manner of Death
Accident    10630
Pending        14
Unknown         9
Name: count, dtype: int64
Lignes : 10654 au départ, 10653 maintenant


**Décision 7 – Regroupement du lieu de décès**
- Quoi : nouvelle colonne `Lieu` en 5 groupes, construite à partir de `Location` (gardée telle quelle).
- Pourquoi : le formulaire a changé en 2021 (`Residence` devient `Decedent's Home`, `Hospital` est découpé en
  sous-catégories). Sans regroupement, les années ne sont pas comparables (constat 9).
- Domicile : Residence, Decedent's Home (2 écritures de l'apostrophe)
- Hôpital : Hospital, ses 3 sous-catégories 2022 et `Hiospital` (faute de frappe).
  `Dead On Arrival` (81) est classé en Hôpital par cohérence avec 2012-2020 : limite, ces décès ont pu survenir avant l'arrivée.
- Autre : Other, Other (Specify)
- Établissement d'hébergement : Nursing Home, Convalescent Home, Hospice, Hospice Facility, Assisted Living, Shelter
- Unknown : les 22 vides

In [9]:
lieux_domicile = ["Residence", "Decedent's Home", "Decedent’s Home"]
lieux_hopital = ["Hospital", "Hospital - ER/Outpatient", "Hospital - Inpatient",
                 "Hospital - Dead On Arrival", "Hiospital"]
lieux_autre = ["Other", "Other (Specify)"]
lieux_hebergement = ["Nursing Home", "Convalescent Home", "Hospice",
                     "Hospice Facility", "Assisted Living", "Shelter"]

In [10]:
deces["Lieu"] = "Unknown"
deces.loc[deces["Location"].isin(lieux_domicile), "Lieu"] = "Domicile"
deces.loc[deces["Location"].isin(lieux_hopital), "Lieu"] = "Hôpital"
deces.loc[deces["Location"].isin(lieux_autre), "Lieu"] = "Autre"
deces.loc[deces["Location"].isin(lieux_hebergement), "Lieu"] = "Établissement d'hébergement"
print(deces["Lieu"].value_counts())

Lieu
Domicile                       5626
Hôpital                        3065
Autre                          1921
Unknown                          22
Établissement d'hébergement      19
Name: count, dtype: int64


**Décision 8 : orthographe des comtés de résidence**

**Constat** : parmi les résidents du Connecticut (État `CT` ou vide), 2 noms de comté sont mal orthographiés : `MIDDLESEZ` (1 ligne) et `NEWLONDON` (1 ligne).

**Solution** : mise en majuscules et suppression des espaces en trop (par sécurité), puis correction des 2 fautes : `MIDDLESEZ` → `MIDDLESEX`, `NEWLONDON` → `NEW LONDON`.

**Pourquoi** : sans correction, ces décès ne seraient rattachés à aucun comté dans la Q4.

**Impact** : 2 lignes corrigées, 0 ligne supprimée. Les 1 163 comtés manquants sont laissés vides : on ne peut pas deviner le comté.

In [11]:
deces["Residence County"] = deces["Residence County"].str.strip().str.upper()
nb_fautes_comtes = deces["Residence County"].isin(["MIDDLESEZ", "NEWLONDON"]).sum()
deces["Residence County"] = deces["Residence County"].str.replace("MIDDLESEZ", "MIDDLESEX")
deces["Residence County"] = deces["Residence County"].str.replace("NEWLONDON", "NEW LONDON")
assert deces["Residence County"].isin(["MIDDLESEZ", "NEWLONDON"]).sum() == 0
print(f"{nb_fautes_comtes} lignes corrigées")

2 lignes corrigées


## Vérification et export
- Une seule ligne supprimée sur tout le nettoyage (le décès `Natural`) : 10 654 → 10 653 lignes.
- 0 doublon strict (constat 11) : aucune suppression nécessaire.
- Fichier produit : `data/processed/deces_nettoyes.csv`, point de départ de `03_analyse.ipynb`.

In [12]:
assert len(deces) == nb_lignes_depart - 1, "Nombre de lignes inattendu"
deces.to_csv("../data/processed/deces_nettoyes.csv", index=False)
print(f"Export terminé : {len(deces)} lignes, {deces.shape[1]} colonnes")

Export terminé : 10653 lignes, 46 colonnes
